In [ ]:
import os, glob
import numpy as np
import pandas as pd
from typing import Dict, List

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.zip

import zipfile, os, torch, numpy as np

zip_name = next(iter(uploaded.keys()))

# unzip into clean folder
extract_root = "/content/train_unzipped"
os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_root)

# find all CSVs anywhere under extract_root
csv_paths = []
for root, dirs, files in os.walk(extract_root):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_paths.append(os.path.join(root, f))

DATA_DIR = os.path.dirname(csv_paths[0]) + "/"

print(f"Found {len(csv_paths)} CSVs")
print("Example CSV:", csv_paths[0])
print("Using DATA_DIR =", DATA_DIR)

# ---------------------------
# Config
# ---------------------------
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 30
HIDDEN = 96
LAYERS = 1
DROPOUT = 0.1
EMB_DIM = 6
N_HEADS = 2
LAMBDA_PHYS = 0.003  # weight for physics regularizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("DEVICE =", DEVICE)


Saving train.zip to train.zip
Found 36 CSVs
Example CSV: /content/train_unzipped/train/train/input_2023_w14.csv
Using DATA_DIR = /content/train_unzipped/train/train/
DEVICE = cuda


In [ ]:
# ---------------------------
# Helper functions
# ---------------------------
def angle_to_sincos(deg: np.ndarray):
    """
    Convert angles in degrees to sin/cos for smoother learning.
    """
    rad = np.deg2rad(deg)
    return np.sin(rad), np.cos(rad)

def load_week_files(prefix: str):
    """
    Load weekly files in sorted order from week 1 to week 18.
    """
    files = sorted(glob.glob(os.path.join(DATA_DIR, f"{prefix}_2023_w*.csv")))
    if not files:
        raise FileNotFoundError(f"No {prefix} files in {DATA_DIR}")
    return files

def build_vocab(series: pd.Series):
    uniq = sorted(series.dropna().unique().tolist())
    stoi = {v: i+1 for i, v in enumerate(uniq)}
    return stoi

def get_week_num(path: str) -> int:
    """
    Extracts week number from filenames like:
    input_2023_w01.csv -> 1
    output_2023_w18.csv -> 18
    """
    base = os.path.basename(path)
    wk = base.split("_w")[1].split(".")[0]
    return int(wk)

# ---------------------------
# Dataset building (player-level)
# ---------------------------

def make_samples(input_df: pd.DataFrame, output_df: pd.DataFrame,
                 vocabs: Dict[str, Dict], norm_stats: Dict) -> List[Dict]:
    """
    One sample per target player per play.
    Tracks only that player's trajectory.
    """
    targets = input_df[input_df["player_to_predict"] == True].copy()
    if len(targets) == 0:
        return []

    # ensure sorting
    targets = targets.sort_values(["game_id","play_id","nfl_id","frame_id"])
    output_df = output_df.sort_values(["game_id","play_id","nfl_id","frame_id"])

    in_groups = targets.groupby(["game_id","play_id","nfl_id"])
    out_groups = output_df.groupby(["game_id","play_id","nfl_id"])

    samples = []
    for key, g_in in in_groups:
        if key not in out_groups.groups:
            continue
        g_out = out_groups.get_group(key)

        # continuous features per frame
        x = g_in["x"].values
        y = g_in["y"].values
        s = g_in["s"].values
        a = g_in["a"].values

        dir_sin, dir_cos = angle_to_sincos(g_in["dir"].values)
        o_sin, o_cos = angle_to_sincos(g_in["o"].values)

        ball_x = g_in["ball_land_x"].values
        ball_y = g_in["ball_land_y"].values
        yardline = g_in["absolute_yardline_number"].values

        # continuous raw
        x = g_in["x"].values
        y = g_in["y"].values
        s = g_in["s"].values
        a = g_in["a"].values

        dir_raw = g_in["dir"].values
        o_raw   = g_in["o"].values

        ball_x = g_in["ball_land_x"].values
        ball_y = g_in["ball_land_y"].values
        yardline = g_in["absolute_yardline_number"].values

        # normalize scalar features using train stats
        def zscore(arr, key):
            m = norm_stats["mean"][key]
            sd = norm_stats["std"][key]
            return (arr - m) / sd

        x = zscore(x, "x")
        y = zscore(y, "y")
        s = zscore(s, "s")
        a = zscore(a, "a")
        ball_x  = zscore(ball_x, "ball_land_x")
        ball_y  = zscore(ball_y, "ball_land_y")
        yardline = zscore(yardline, "absolute_yardline_number")

        # directional features still go through sin/cos
        dir_sin, dir_cos = angle_to_sincos(dir_raw)
        o_sin,   o_cos   = angle_to_sincos(o_raw)


        cont = np.stack(
            [x, y, s, a, dir_sin, dir_cos, o_sin, o_cos, ball_x, ball_y, yardline],
            axis=1
        ).astype(np.float32)

        # categorical
        def map_cat(col, vocab):
            val = g_in[col].iloc[0] if col in g_in else None
            return vocab.get(val, 0)

        cat = np.array([
            map_cat("player_position", vocabs["player_position"]),
            map_cat("player_side",     vocabs["player_side"]),
            map_cat("player_role",     vocabs["player_role"]),
            map_cat("play_direction",  vocabs["play_direction"]),
        ], dtype=np.int64)

        # target trajectory
        y_future = g_out[["x","y"]].values.astype(np.float32)
        num_out = int(g_in["num_frames_output"].iloc[0])
        if len(y_future) != num_out:
            y_future = y_future[:num_out]

        samples.append({
            "cont": cont,
            "cat": cat,
            "y_future": y_future,
            "num_out": num_out
        })

    return samples


class BDBDataset(Dataset):
    def __init__(self, samples: List[Dict]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        return (
            torch.from_numpy(s["cont"]),
            torch.from_numpy(s["cat"]),
            torch.from_numpy(s["y_future"]),
            s["num_out"]
        )


def collate_fn(batch):
    """
    Pads variable T_in and variable T_out in a batch.
    """
    cont_list, cat_list, y_list, nout_list = zip(*batch)

    Tin = [c.shape[0] for c in cont_list]
    Tout = [y.shape[0] for y in y_list]
    max_Tin = max(Tin)
    max_Tout = max(Tout)

    cont_dim = cont_list[0].shape[1]
    cat_dim  = cat_list[0].shape[0]

    cont_pad = torch.zeros(len(batch), max_Tin, cont_dim)
    tin_mask = torch.zeros(len(batch), max_Tin)

    y_pad    = torch.zeros(len(batch), max_Tout, 2)
    tout_mask = torch.zeros(len(batch), max_Tout)

    cat_tensor = torch.zeros(len(batch), cat_dim, dtype=torch.long)

    for i, (c, cat, y) in enumerate(zip(cont_list, cat_list, y_list)):
        cont_pad[i, :c.shape[0]] = c
        tin_mask[i, :c.shape[0]] = 1.0

        y_pad[i, :y.shape[0]] = y
        tout_mask[i, :y.shape[0]] = 1.0

        cat_tensor[i] = cat

    return cont_pad, tin_mask, cat_tensor, y_pad, tout_mask

# ---------------------------
# Physics-informed Transformer model
# ---------------------------

class PhysicsInformedTrajectoryTransformer(nn.Module):
    """
    Physics-informed trajectory model:
    - Transformer encoder over pre-throw frames (per player)
    - Decoder predicts accelerations (ax, ay) autoregressively
    - Positions are obtained by integrating velocity + acceleration
    """
    def __init__(self, cont_dim: int, cat_sizes,
                 hidden=HIDDEN, layers=LAYERS,
                 dropout=DROPOUT, emb_dim=EMB_DIM, n_heads=N_HEADS):
        super().__init__()

        # categorical embeddings
        self.emb_pos  = nn.Embedding(cat_sizes["player_position"], emb_dim)
        self.emb_side = nn.Embedding(cat_sizes["player_side"], emb_dim)
        self.emb_role = nn.Embedding(cat_sizes["player_role"], emb_dim)
        self.emb_dir  = nn.Embedding(cat_sizes["play_direction"], emb_dim)
        self.cat_dim = emb_dim * 4

        # input projection
        self.input_dim  = cont_dim + self.cat_dim
        self.input_proj = nn.Linear(self.input_dim, hidden)
        # time embeddings
        self.max_time = 300
        self.time_embed = nn.Embedding(self.max_time, hidden)


        # temporal transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)

        # physics-aware decoder: GRUCell that outputs accelerations
        # decoder input = [xy, v, cat_embed, enc_summary]
        self.dec_in_dim = 2 + 2 + self.cat_dim + hidden
        self.decoder_cell = nn.GRUCell(self.dec_in_dim, hidden)

        self.acc_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2)
        )

        # learnable timestep scaling
        self.dt_log = nn.Parameter(torch.zeros(1))

    def forward(self, cont, tin_mask, cat, max_Tout):
        """
        cont: [B, T_in, cont_dim]
        tin_mask: [B, T_in]
        cat: [B,4]
        max_Tout: int
        """
        B, T_in, _ = cont.shape
        device = cont.device

        # categorical embedding
        e = torch.cat([
            self.emb_pos(cat[:,0]),
            self.emb_side(cat[:,1]),
            self.emb_role(cat[:,2]),
            self.emb_dir(cat[:,3]),
        ], dim=-1)

        e_in = e.unsqueeze(1).expand(B, T_in, self.cat_dim)
        enc_in = torch.cat([cont, e_in], dim=-1)

        x = self.input_proj(enc_in)

        # add time embeddings
        T_in = cont.shape[1]
        device = cont.device
        time_idx = torch.arange(T_in, device=device).unsqueeze(0).expand(B, T_in)
        x = x + self.time_embed(time_idx)
        src_key_padding_mask = (tin_mask == 0)
        x_enc = self.encoder(x, src_key_padding_mask=src_key_padding_mask)

        # sequence lengths and last valid index
        lengths = tin_mask.sum(dim=1).long()
        lengths = torch.clamp(lengths, min=1)
        idx = torch.arange(B, device=device)

        enc_summary = x_enc[idx, lengths-1]

        # initial physics state: last position + velocity
        last_xy = cont[idx, lengths-1, 0:2]

        # cont columns: [x,y,s,a,dir_sin,dir_cos,o_sin,o_cos,ball_x,ball_y,yardline]
        s_last        = cont[idx, lengths-1, 2]
        dir_sin_last  = cont[idx, lengths-1, 4]
        dir_cos_last  = cont[idx, lengths-1, 5]

        vx0 = s_last * dir_cos_last
        vy0 = s_last * dir_sin_last
        v_last = torch.stack([vx0, vy0], dim=-1)   # [B,2]

        # decoder hidden state initialized from encoder summary
        h = enc_summary

        dt = torch.exp(self.dt_log)

        xy_t = last_xy
        v_t  = v_last

        preds = []
        cat_rep = e

        for t in range(max_Tout):
            dec_in = torch.cat([
                xy_t,       # [B,2]
                v_t,        # [B,2]
                cat_rep,    # [B,cat_dim]
                enc_summary # [B,hidden]
            ], dim=-1)      # [B, dec_in_dim]

            h = self.decoder_cell(dec_in, h)
            a_t = self.acc_head(h)

            # integrate velocity & position
            v_t  = v_t + a_t * dt
            xy_t = xy_t + v_t * dt

            preds.append(xy_t.unsqueeze(1))

        pred_xy = torch.cat(preds, dim=1)
        return pred_xy

# ---------------------------
# Losses
# ---------------------------

def masked_rmse_loss(pred, target, mask):
    """
    pred,target: [B,T,2], mask: [B,T]
    Matches Kaggle RMSE definition over all valid (x,y) points.
    """
    diff2 = (pred - target)**2
    diff2 = diff2 * mask.unsqueeze(-1)
    mse = diff2.sum() / (mask.sum()*2.0 + 1e-8)
    return torch.sqrt(mse + 1e-8)


def physics_regularizer(pred_xy, tout_mask, dt=1.0,
                        max_speed=12.0,   # yards/s
                        max_acc=8.0):     # yards/s^2
    """
    pred_xy: [B,T,2], tout_mask: [B,T] (1=valid, 0=pad)
    Penalize speeds / accels that exceed physical thresholds.
    """
    B, T, _ = pred_xy.shape
    if T < 3:
        return torch.tensor(0.0, device=pred_xy.device)

    # velocity
    v = (pred_xy[:,1:] - pred_xy[:,:-1]) / dt          # [B,T-1,2]
    mask_v = tout_mask[:,1:] * tout_mask[:,:-1]        # [B,T-1]

    speed = torch.linalg.norm(v, dim=-1)               # [B,T-1]
    speed_excess = torch.clamp(speed - max_speed, min=0.0)
    speed_pen = (speed_excess**2 * mask_v).sum() / (mask_v.sum() + 1e-8)

    # acceleration
    a = (v[:,1:] - v[:,:-1]) / dt                      # [B,T-2,2]
    mask_a = mask_v[:,1:] * mask_v[:,:-1]              # [B,T-2]

    acc_mag = torch.linalg.norm(a, dim=-1)             # [B,T-2]
    acc_excess = torch.clamp(acc_mag - max_acc, min=0.0)
    acc_pen = (acc_excess**2 * mask_a).sum() / (mask_a.sum() + 1e-8)

    return speed_pen + acc_pen

# ---------------------------
# Train
# ---------------------------

def main():
    in_files = load_week_files("input")
    out_files = load_week_files("output")

    # split by week (1–13 train, 14–18 val)
    train_pairs, val_pairs = [], []
    for fin, fout in zip(in_files, out_files):
        w = get_week_num(fin)
        if w <= 13:
            train_pairs.append((fin, fout))
        else:
            val_pairs.append((fin, fout))

    print("Train weeks:", [get_week_num(p[0]) for p in train_pairs])
    print("Val weeks:",   [get_week_num(p[0]) for p in val_pairs])

    # build vocabs from TRAIN weeks only
    train_in_df = pd.concat(
        [pd.read_csv(fin) for fin, _ in train_pairs],
        ignore_index=True
    )

    # feature normalization stats (train only)
    cont_cols = [
        "x", "y", "s", "a",
        "dir", "o",
        "ball_land_x", "ball_land_y",
        "absolute_yardline_number"
    ]

    feat_means = train_in_df[cont_cols].mean()
    feat_stds  = train_in_df[cont_cols].std().replace(0.0, 1.0)

    norm_stats = {"mean": feat_means.to_dict(), "std": feat_stds.to_dict()}


    vocabs = {
        "player_position": build_vocab(train_in_df["player_position"]),
        "player_side":     build_vocab(train_in_df["player_side"]),
        "player_role":     build_vocab(train_in_df["player_role"]),
        "play_direction":  build_vocab(train_in_df["play_direction"]),
    }
    cat_sizes = {k: len(v)+1 for k, v in vocabs.items()}

    # build samples
    train_samples, val_samples = [], []
    for fin, fout in train_pairs:
        inp = pd.read_csv(fin)
        out = pd.read_csv(fout)
        train_samples += make_samples(inp, out, vocabs, norm_stats)

    for fin, fout in val_pairs:
        inp = pd.read_csv(fin)
        out = pd.read_csv(fout)
        val_samples += make_samples(inp, out, vocabs, norm_stats)

    print("Total TRAIN target-player samples:", len(train_samples))
    print("Total VAL target-player samples:", len(val_samples))

    # dataloaders
    train_loader = DataLoader(
        BDBDataset(train_samples),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )
    val_loader = DataLoader(
        BDBDataset(val_samples),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    # model
    cont_dim = train_samples[0]["cont"].shape[1]   # 11
    model = PhysicsInformedTrajectoryTransformer(cont_dim, cat_sizes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    # training loop
    for ep in range(1, EPOCHS + 1):
        model.train()
        tr_loss, tr_data_loss, tr_phys_loss = 0.0, 0.0, 0.0

        for b_idx, (cont, tin_mask, cat, y, tout_mask) in enumerate(train_loader):
            if b_idx % 50 == 0:
                print(f"[Epoch {ep}] Batch {b_idx}/{len(train_loader)}")

            cont, tin_mask  = cont.to(DEVICE), tin_mask.to(DEVICE)
            cat, y, tout_mask = cat.to(DEVICE), y.to(DEVICE), tout_mask.to(DEVICE)

            pred = model(cont, tin_mask, cat, y.shape[1])

            data_loss = masked_rmse_loss(pred, y, tout_mask)
            phys_loss = physics_regularizer(pred, tout_mask)

            loss = data_loss + LAMBDA_PHYS * phys_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

            tr_loss      += loss.item()
            tr_data_loss += data_loss.item()
            tr_phys_loss += phys_loss.item()

        n_batches = len(train_loader)
        tr_loss      /= n_batches
        tr_data_loss /= n_batches
        tr_phys_loss /= n_batches

        # validation
        model.eval()
        va_loss, va_data_loss, va_phys_loss = 0.0, 0.0, 0.0
        with torch.no_grad():
            for cont, tin_mask, cat, y, tout_mask in val_loader:
                cont, tin_mask  = cont.to(DEVICE), tin_mask.to(DEVICE)
                cat, y, tout_mask = cat.to(DEVICE), y.to(DEVICE), tout_mask.to(DEVICE)

                pred = model(cont, tin_mask, cat, y.shape[1])

                data_loss = masked_rmse_loss(pred, y, tout_mask)
                phys_loss = physics_regularizer(pred, tout_mask)
                loss = data_loss + LAMBDA_PHYS * phys_loss

                va_loss      += loss.item()
                va_data_loss += data_loss.item()
                va_phys_loss += phys_loss.item()

        n_vb = len(val_loader)
        va_loss      /= n_vb
        va_data_loss /= n_vb
        va_phys_loss /= n_vb

        print(
            f"Epoch {ep}: "
            f"train_RMSE+phys={tr_loss:.4f} (data={tr_data_loss:.4f}, phys={tr_phys_loss:.4f}) | "
            f"val_RMSE+phys={va_loss:.4f} (data={va_data_loss:.4f}, phys={va_phys_loss:.4f})"
        )

    # save
    torch.save(
        {"model": model.state_dict(), "vocabs": vocabs, "norm_stats": norm_stats},
        "bdb2026_phys_transformer.pt"
    )

    print("Saved bdb2026_phys_transformer.pt")


if __name__ == "__main__":
    main()

Train weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Val weeks: [14, 15, 16, 17, 18]
Total TRAIN target-player samples: 32856
Total VAL target-player samples: 13189
[Epoch 1] Batch 0/257
[Epoch 1] Batch 50/257
[Epoch 1] Batch 100/257
[Epoch 1] Batch 150/257
[Epoch 1] Batch 200/257
[Epoch 1] Batch 250/257


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1: train_RMSE+phys=16.2971 (data=16.2506, phys=15.4986) | val_RMSE+phys=6.1178 (data=5.9916, phys=42.0753)
[Epoch 2] Batch 0/257
[Epoch 2] Batch 50/257
[Epoch 2] Batch 100/257
[Epoch 2] Batch 150/257
[Epoch 2] Batch 200/257
[Epoch 2] Batch 250/257
Epoch 2: train_RMSE+phys=4.4169 (data=4.3533, phys=21.2088) | val_RMSE+phys=3.2639 (data=3.2473, phys=5.5280)
[Epoch 3] Batch 0/257
[Epoch 3] Batch 50/257
[Epoch 3] Batch 100/257
[Epoch 3] Batch 150/257
[Epoch 3] Batch 200/257
[Epoch 3] Batch 250/257
Epoch 3: train_RMSE+phys=2.8997 (data=2.8928, phys=2.3075) | val_RMSE+phys=2.5162 (data=2.5143, phys=0.6189)
[Epoch 4] Batch 0/257
[Epoch 4] Batch 50/257
[Epoch 4] Batch 100/257
[Epoch 4] Batch 150/257
[Epoch 4] Batch 200/257
[Epoch 4] Batch 250/257
Epoch 4: train_RMSE+phys=2.3619 (data=2.3606, phys=0.4273) | val_RMSE+phys=2.0763 (data=2.0757, phys=0.2094)
[Epoch 5] Batch 0/257
[Epoch 5] Batch 50/257
[Epoch 5] Batch 100/257
[Epoch 5] Batch 150/257
[Epoch 5] Batch 200/257
[Epoch 5] Batch 250